# Chapitre 4 — Implémentation des modèles

**TER ISFA — Modèle de Merton vs Machine Learning dans la prédiction du risque de crédit**  
Master 1 Actuariat — Année 2025/2026

---

## Vue d'ensemble du notebook

Ce notebook implémente et compare trois approches de prédiction du risque de crédit sur le jeu de données *Polish Companies Bankruptcy* (sous-ensemble Année 5, 5 910 firmes, 6.94 % de défauts).

### Plan du notebook

| Section | Contenu |
|---|---|
| **1. Imports et configuration** | Chargement des librairies, palette ISFA |
| **2. Chargement et prétraitement** | Pipeline complet sans fuite de données |
| **3. Modèle de Merton — MLE** | Calibration, DD, PD, spread |
| **4. Régression Logistique** | Cross-validation, entraînement, coefficients |
| **5. XGBoost** | Random search, entraînement, importance des variables |
| **6. Ablation study** | Isole la vraie contribution des features Merton |
| **7. Comparaison finale** | Tableau des 4 métriques, courbes ROC, distributions |
| **8. Sauvegarde** | Exports CSV et PDF pour le rapport LaTeX |


**Trois modèles :**
1. **Merton** : calibration MLE, Distance au Défaut (DD), probabilité de défaut risque-neutre PD
2. **Régression Logistique** : régularisation L1/L2, validation croisée stratifiée
3. **XGBoost** : Random Search des hyperparamètres, importance des variables

### Fichiers requis
- `5year.arff` — données avec variable cible (0/1)
- `X.csv` — features brutes toutes années (optionnel pour contexte)

### Installation
```bash
pip install xgboost imbalanced-learn scipy scikit-learn pandas numpy matplotlib seaborn
```

---
## 1. Imports et configuration

On charge toutes les librairies en une seule cellule pour garder le notebook propre.  
On définit aussi la palette de couleurs ISFA utilisée dans toutes les figures.

In [ ]:
# ── Librairies standard ───────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Chargement du fichier .arff ───────────────────────────────
from scipy.io import arff

# ── Fonctions mathématiques pour Merton ──────────────────────
from scipy.stats import norm        # loi normale N(x)
from scipy import optimize          # solveur pour le système non linéaire

# ── Scikit-learn : preprocessing et modèles ──────────────────
from sklearn.model_selection import (
    train_test_split,         # découpage train/test
    StratifiedKFold,          # validation croisée stratifiée
    RandomizedSearchCV        # random search des hyperparamètres
)
from sklearn.impute import SimpleImputer       # pour l'imputation médiane
from sklearn.preprocessing import StandardScaler  # pour la standardisation
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone                 # copier un modèle sans ses paramètres
from sklearn.metrics import (
    roc_auc_score,            # AUC-ROC
    f1_score,                 # F1-Score
    brier_score_loss,         # Score de Brier (calibration)
    roc_curve,                # courbe ROC complète
    confusion_matrix,
    classification_report
)

# ── XGBoost (optionnel) ───────────────────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost {xgb.__version__} disponible')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost absent — substitut RandomForest utilisé')
    print('Installer avec : pip install xgboost')

# ── SMOTE : rééchantillonnage de la classe minoritaire ────────
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print('imbalanced-learn disponible')
except ImportError:
    SMOTE_AVAILABLE = False
    print('imbalanced-learn absent — entraînement sans SMOTE')
    print('Installer avec : pip install imbalanced-learn')

# ── Palette de couleurs  ──────────────────────────────────
BLUE   = '#003366'   # bleu   → firmes non défaillantes
PURPLE = '#500078'   # violet  → firmes défaillantes
GRAY   = '#646464'   # gris  → modèle Merton / référence

# ── Paramètres globaux matplotlib ────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 110,
    'font.family'       : 'serif',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)

print('\nConfiguration OK')

---
## 2. Chargement et prétraitement des données

### Règle fondamentale : aucune fuite de données

Tout le preprocessing (winsorisation, imputation, standardisation) doit être **ajusté uniquement sur l'ensemble d'entraînement** puis appliqué à l'ensemble de test.  
Appliquer ces transformations sur les données complètes *avant* le split introduit une fuite d'information (*data leakage*) : le modèle "verrait" des informations du test pendant l'entraînement, ce qui surestime ses performances.

La même règle s'applique aux features Merton : elles doivent être calculées **après** le split, séparément sur train et test.

### Ordre strict du pipeline
```
1. Chargement → 2. Split stratifié 80/20 → 3. Winsorisation (fit sur train)
→ 4. Imputation médiane (fit sur train) → 5. Standardisation (fit sur train)
→ 6. SMOTE (sur train uniquement) → 7. Features Merton (séparées train/test)
```

In [ ]:
# ── Chargement du fichier 5year.arff ─────────────────────────
# Le fichier .arff contient 64 ratios financiers + la variable cible 'class'
raw, meta = arff.loadarff('5year.arff')
df = pd.DataFrame(raw)

# La variable cible est encodée en bytes dans le format ARFF
df['class'] = df['class'].apply(lambda x: int(x.decode('utf-8')))

# Renommer Attr1..Attr64 en A1..A64 pour avoir une cohérence avec le rapport
df.rename(columns={f'Attr{i}': f'A{i}' for i in range(1, 65)}, inplace=True)

print(f'Dataset chargé : {df.shape[0]:,} observations x {df.shape[1]} colonnes')
print(f'Défauts (class=1) : {df["class"].sum()} ({df["class"].mean()*100:.2f}%)')

In [ ]:
# ── Variables sélectionnées ───────────────────────────────────
# 12 ratios financiers couvrant rentabilité, levier, liquidité, taille
# Sélectionnés pour leur pertinence économique (lien avec le modèle de Merton)
# et leur pouvoir prédictif documenté dans la littérature
SELECTED = ['A1',   # Résultat net / Actif total       → drift μ
            'A2',   # Total passif / Actif total        → levier ℓ_t
            'A3',   # Fonds de roulement / Actif total  → liquidité CT
            'A4',   # Actifs courants / Passif CT       → ratio de liquidité
            'A7',   # EBIT / Actif total                → rentabilité opérationnelle
            'A10',  # Capitaux propres / Actif total    → solvabilité (1 - ℓ_t)
            'A16',  # (Bénéfice brut + amort.) / passif → capacité de remboursement
            'A17',  # Actif total / Total passif        → levier inverse
            'A18',  # Bénéfice brut / Actif total       → marge brute
            'A29',  # log(Actif total)                  → taille de la firme
            'A36',  # CA / Actif total                  → rotation des actifs
            'A40']  # (Actifs courants - stocks) / CT   → liquidité immédiate

X = df[SELECTED].copy()
y = df['class'].copy()

print(f'Matrice de features : {X.shape}')
print(f'Valeurs manquantes sur les 12 variables : {X.isnull().sum().sum()}')

In [ ]:
# ── Fonctions de winsorisation ────────────────────────────────
# La winsorisation écrête les valeurs extrêmes causées par des dénominateurs
# proches de zéro dans les ratios financiers (ex : A2 max brut = 72.4)
# On coupe aux percentiles P1 et P99 calculés sur l'ensemble d'entraînement

def fit_winsorise(df_train, lo=0.01, hi=0.99):
    """Calcule les bornes P1/P99 sur df_train pour chaque colonne."""
    return {
        col: (df_train[col].quantile(lo), df_train[col].quantile(hi))
        for col in df_train.columns
    }

def apply_winsorise(df_in, bounds):
    """Applique les bornes pré-calculées à df_in."""
    out = df_in.copy()
    for col, (lo, hi) in bounds.items():
        if col in out.columns:
            out[col] = out[col].clip(lo, hi)
    return out

print('Fonctions de winsorisation définies.')

In [ ]:
# ── Découpage train / test (80 % / 20 %) ─────────────────────
# Le découpage est STRATIFIÉ : le taux de défaut de 6.94 % est préservé
# dans les deux ensembles. Tous les paramètres de preprocessing seront
# estimés sur X_train puis appliqués à X_test sans re-ajustement.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # garantit le même taux de défaut dans train et test
)

print('Découpage train/test :')
print(f'  Train : {len(X_train):,} obs — défauts : {y_train.sum()} ({y_train.mean()*100:.2f}%)')
print(f'  Test  : {len(X_test):,}  obs — défauts : {y_test.sum()}  ({y_test.mean()*100:.2f}%)')

In [ ]:
# ── Winsorisation → Imputation → Standardisation ──────────────
# Tout est ajusté sur X_train, appliqué sur X_train ET X_test

# Étape 1 : winsorisation P1-P99
win_bounds   = fit_winsorise(X_train)
X_train_w    = apply_winsorise(X_train, win_bounds)
X_test_w     = apply_winsorise(X_test,  win_bounds)

# Étape 2 : imputation des valeurs manquantes par la médiane
# La médiane est plus robuste que la moyenne pour des distributions asymétriques
imputer      = SimpleImputer(strategy='median')
X_train_imp  = pd.DataFrame(
    imputer.fit_transform(X_train_w),  # fit sur train
    columns=SELECTED, index=X_train_w.index
)
X_test_imp   = pd.DataFrame(
    imputer.transform(X_test_w),       # transform seulement sur test
    columns=SELECTED, index=X_test_w.index
)

# Étape 3 : standardisation (μ=0, σ=1)
# Nécessaire pour la Régression Logistique car L1/L2 pénalisent tous les
# coefficients de la même manière — sans ça, les grandes échelles dominent.
# XGBoost n'en a PAS besoin (arbres invariants à l'échelle).
scaler       = StandardScaler()
X_train_sc   = pd.DataFrame(
    scaler.fit_transform(X_train_imp),
    columns=SELECTED, index=X_train_imp.index
)
X_test_sc    = pd.DataFrame(
    scaler.transform(X_test_imp),
    columns=SELECTED, index=X_test_imp.index
)

print('Preprocessing terminé.')
print(f'  Valeurs manquantes après imputation — train : {X_train_imp.isnull().sum().sum()}')
print(f'  Valeurs manquantes après imputation — test  : {X_test_imp.isnull().sum().sum()}')
print(f'  Moyenne A1 après standardisation (doit ≈ 0) : {X_train_sc["A1"].mean():.6f}')

In [ ]:
# ── SMOTE : rééquilibrage de la classe minoritaire ────────────
# Le taux de défaut de 6.94 % crée un fort déséquilibre de classes.
# SMOTE génère des observations synthétiques de la classe minoritaire
# par interpolation entre observations existantes dans l'espace des features.
# IMPORTANT : on applique uuniquement  sur le train. Le test reste intact.

if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=42)

    # Version non standardisée (pour XGBoost)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_imp, y_train)
    X_train_sm = pd.DataFrame(X_train_sm, columns=SELECTED)

    # Version standardisée (pour Régression Logistique)
    scaler_sm    = StandardScaler()
    X_train_sm_sc = pd.DataFrame(
        scaler_sm.fit_transform(X_train_sm), columns=SELECTED)
    X_test_sm_sc  = pd.DataFrame(
        scaler_sm.transform(X_test_imp), columns=SELECTED)

    vc_avant = pd.Series(y_train).value_counts().sort_index()
    vc_apres = pd.Series(y_train_sm).value_counts().sort_index()
    print('SMOTE appliqué sur le train uniquement :')
    print(f'  Avant  — non-déf : {vc_avant[0]:,}, déf : {vc_avant[1]:,}')
    print(f'  Après  — non-déf : {vc_apres[0]:,}, déf : {vc_apres[1]:,}')

else:
    # Si SMOTE non disponible, on utilise les données brutes
    X_train_sm, y_train_sm = X_train_imp.copy(), y_train.copy()
    X_train_sm_sc = X_train_sc.copy()
    X_test_sm_sc  = X_test_sc.copy()
    print('SMOTE non disponible — entraînement sans rééchantillonnage.')

---
## 3. Modèle de Merton — Calibration par Maximum de Vraisemblance (MLE)

### Principe

Le modèle de Merton traite l'action d'une firme comme un **call sur ses actifs** :

$$E_t = V_t N(d_1) - L_t N(d_2) \qquad \text{avec} \qquad d_2 = \frac{\ln(V_t/L) + (\mu_V - \sigma_V^2/2)T}{\sigma_V\sqrt{T}}$$

Les deux paramètres non observables ($V_t$ et $\sigma_V$) sont estimés par MLE à partir de la série temporelle des capitaux propres.

### Adaptation au dataset polonais

Le dataset ne contient pas de prix boursiers — il n'y a que des données comptables annuelles. On approxime donc :
- **Actif total** $\approx e^{A29}$ (via le logarithme A29)
- **Dette** $\approx A2 \times e^{A29}$ (via le levier A2)
- **Capitaux propres** $\approx (1-A2) \times e^{A29}$

### Règle de non-contamination

Les features Merton sont calculées **après le split**, séparément sur train et test. Calculer les features Merton sur l'ensemble complet puis spliter introduirait une fuite de données.

In [ ]:
# ── Formules de base du modèle de Merton ─────────────────────

def merton_d1d2(V, L, sigma_V, r=0.02, T=1.0):
    """
    Calcule d1 et d2 de la formule de Merton.
    V     : valeur des actifs
    L     : valeur nominale de la dette
    sigma_V : volatilité des actifs
    r     : taux sans risque annuel
    T     : horizon (1 an)
    """
    if V <= 0 or L <= 0 or sigma_V < 0.001:
        return np.nan, np.nan
    d1 = (np.log(V / L) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)
    return d1, d2


def merton_equity(V, L, sigma_V, r=0.02, T=1.0):
    """
    Valeur théorique des capitaux propres selon Black-Scholes.
    C'est un call sur les actifs de la firme avec strike = L.
    """
    d1, d2 = merton_d1d2(V, L, sigma_V, r, T)
    if np.isnan(d1):
        return 0.0
    return V * norm.cdf(d1) - L * np.exp(-r * T) * norm.cdf(d2)


def merton_inversion(E_obs, L, sigma_V, r=0.02, T=1.0,
                     tol=1e-8, max_iter=100):
    """
    Inverse la formule E = f(V, sigma_V) par Newton-Raphson.
    Pour une valeur observée des capitaux propres E_obs,
    trouve la valeur implicite des actifs V_t.
    Initialisation : V_0 = E + L (borne supérieure naturelle).
    """
    V = E_obs + L  # initialisation
    for _ in range(max_iter):
        E_pred = merton_equity(V, L, sigma_V, r, T)
        d1, _  = merton_d1d2(V, L, sigma_V, r, T)
        if np.isnan(d1) or norm.cdf(d1) < 1e-12:
            break
        # Dérivée dE/dV = N(d1) (delta du call)
        V = V - (E_pred - E_obs) / norm.cdf(d1)
        if abs(E_pred - E_obs) < tol:
            break
    return max(V, 1e-6)


print('Fonctions Merton définies : merton_d1d2, merton_equity, merton_inversion')

In [ ]:
# ── Log-vraisemblance MLE avec correction jacobienne ─────────
# La log-vraisemblance standard des log-rendements est augmentée
# du terme jacobien log|∂E/∂V| = log Φ(d1) qui corrige le changement
# de variable E → V. Omettre ce terme biaise l'estimateur.

def merton_loglik(params, equity_series, debt_series, r=0.02, T=1.0):
    """
    Retourne la log-vraisemblance NÉGATIVE (pour minimisation).
    params         : [mu_V, sigma_V]
    equity_series  : série temporelle des capitaux propres E_t
    debt_series    : série temporelle de la dette L_t
    """
    mu_V, sigma_V = params
    if sigma_V <= 0.01:
        return 1e10  # valeur pénalisante si sigma invalide

    dt = 1.0  # données annuelles → Δt = 1 an

    # Étape 1 : inverser E → V pour chaque période
    V_series = []
    log_jacob = 0.0
    for E_t, L_t in zip(equity_series, debt_series):
        if E_t <= 0 or L_t <= 0:
            return 1e10
        V_t = merton_inversion(E_t, L_t, sigma_V, r, T)
        V_series.append(V_t)
        d1, _ = merton_d1d2(V_t, L_t, sigma_V, r, T)
        if np.isnan(d1):
            return 1e10
        # Terme jacobien : log N(d1) pour chaque observation
        log_jacob += np.log(max(norm.cdf(d1), 1e-12))

    V_series    = np.array(V_series)
    log_returns = np.diff(np.log(V_series))  # log-rendements des actifs
    n           = len(log_returns)
    if n == 0:
        return 1e10

    mu_adj = mu_V - 0.5 * sigma_V**2  # dérive ajustée

    # Log-vraisemblance gaussienne des log-rendements + jacobien
    ll = (
        - n * np.log(sigma_V * np.sqrt(2 * np.pi * dt))
        - np.sum(np.log(V_series[:-1] + 1e-12))    # terme de normalisation
        - np.sum((log_returns - mu_adj * dt)**2)    # terme quadratique
          / (2 * sigma_V**2 * dt)
        + log_jacob                                 # correction jacobienne
    )
    return -ll  # on retourne le négatif pour minimiser


print('Fonction merton_loglik définie.')

In [ ]:
# ── Calibration Merton par firme (cross-sectionnel) ───────────
# En l'absence de séries temporelles par firme dans le dataset polonais,
# on utilise une calibration cross-sectionnelle :
# pour chaque observation, on résout le système de Merton
# {E_t = V*N(d1) - L*N(d2), σ_E*E = σ_V*V*N(d1)} par fsolve.

def compute_merton_row(row, r=0.02, T=1.0):
    """
    Calibre Merton pour une observation unique.
    Retourne un dict avec DD, PD, sigma_V, spread.
    """
    # Reconstruction des variables Merton depuis les ratios comptables
    lev  = np.clip(row['A2'] if not np.isnan(row['A2']) else 0.5, 0.01, 0.99)
    size = row['A29'] if not np.isnan(row['A29']) else 4.0
    V    = np.exp(size)      # actif total ≈ exp(A29)
    L    = lev * V           # dette ≈ A2 × actif total
    E    = (1 - lev) * V     # capitaux propres = actif - dette

    # Initialisation de sigma_V depuis la rentabilité
    roa     = row['A1'] if not np.isnan(row['A1']) else 0.0
    sigma_E = max(abs(roa) * 2 + 0.15, 0.10)   # proxy de la vol. des capitaux propres
    sigma_V = max(sigma_E * E / (E + L), 0.05)  # vol. des actifs (borne inférieure)

    # Résoudre le système non linéaire de Merton par fsolve
    def system(sv):
        if sv[0] <= 0:
            return [1e6, 1e6]
        d1, d2 = merton_d1d2(V, L, sv[0], r, T)
        if np.isnan(d1):
            return [1e6, 1e6]
        E_impl = V * norm.cdf(d1) - L * np.exp(-r*T) * norm.cdf(d2)
        eq1    = E_impl - E                       # équation 1 : E_impl = E_obs
        eq2    = sigma_E * E - sv[0] * V * norm.cdf(d1)  # équation 2 : σ_E*E = σ_V*V*N(d1)
        return [eq1, eq2]

    try:
        sol = optimize.fsolve(system, [sigma_V], full_output=True)
        if sol[2] == 1:  # convergence
            sigma_V = max(float(sol[0][0]), 0.01)
    except Exception:
        pass  # garder l'initialisation si fsolve échoue

    # Calcul de la Distance au Défaut et de la PD risque-neutre
    d1, d2 = merton_d1d2(V, L, sigma_V, r, T)
    DD     = d2 if not np.isnan(d2) else 0.0
    PD     = float(norm.cdf(-DD))     # PD risque-neutre = N(-d2)

    # Calcul du spread de crédit implicite
    Lt    = L * np.exp(-r * T)        # valeur actualisée de la dette
    lev_t = Lt / V if V > 0 else 0.5 # levier actualisé
    spread = 0.0
    if lev_t > 0:
        h1 = (np.log(1/lev_t) + sigma_V**2*T/2) / (sigma_V*np.sqrt(T))
        h2 = h1 - sigma_V * np.sqrt(T)
        arg = norm.cdf(-h1) / lev_t + norm.cdf(h2)
        spread = float(-np.log(max(arg, 1e-12)) / T)

    return {'DD': DD, 'PD': PD, 'sigma_V': sigma_V, 'spread': spread}


print('Fonction compute_merton_row définie.')

In [ ]:
# ── Application : calcul des features Merton sur train ET test ─
# RÈGLE CLEF : on appelle compute_merton_row séparément sur les
# observations du train et celles du test. Aucune information
# du test ne contammine le calcul des features du train.

print('Calibration Merton sur le train...')
mf_train = (
    df.loc[X_train.index, SELECTED]
    .apply(compute_merton_row, axis=1, result_type='expand')
    .reset_index(drop=True)
)

print('Calibration Merton sur le test...')
mf_test = (
    df.loc[X_test.index, SELECTED]
    .apply(compute_merton_row, axis=1, result_type='expand')
    .reset_index(drop=True)
)

MERTON_COLS = ['DD', 'PD', 'sigma_V', 'spread']

print(f'\nFeatures Merton — train : {mf_train.shape}, test : {mf_test.shape}')
print(f'Valeurs manquantes — train : {mf_train.isnull().sum().sum()}')
print('\nStatistiques des features Merton (train) :')
mf_train.describe().round(3)

In [ ]:
# ── Score AUC du modèle de Merton seul ───────────────────────
# On évalue directement le score N(-DD) sur le test set.
# Plus DD est élevé → firme loin du défaut → score de risque faible.
# Le score de risque est donc PD = N(-DD).

merton_prob_test = mf_test['PD'].fillna(0.5).values
auc_merton = roc_auc_score(y_test.values, merton_prob_test)
print(f'AUC-ROC du modèle de Merton seul (score brut N(-DD)) : {auc_merton:.4f}')

In [ ]:
# ── Distribution de DD par statut de défaut ───────────────────
# Ce graphique montre visuellement pourquoi l'AUC de Merton est limitée :
# les distributions DD des firmes défaillantes et non défaillantes
# se chevauchent fortement.

# Récupérer les indices train pour associer y_train et mf_train
dd_values   = mf_train['DD'].fillna(0).values
y_train_arr = y_train.values

fig, ax = plt.subplots(figsize=(10, 4))

d0 = dd_values[y_train_arr == 0]   # non-défaillants
d1 = dd_values[y_train_arr == 1]   # défaillants

# Winsoriser pour la visualisation
p1, p99 = np.percentile(dd_values, [1, 99])
bins = np.linspace(p1, p99, 30)

ax.hist(np.clip(d0, p1, p99), bins=bins, color=BLUE,   alpha=0.55, density=True,
        edgecolor='white', linewidth=0.3, label=f'Non-défaillants (n={len(d0):,})')
ax.hist(np.clip(d1, p1, p99), bins=bins, color=PURPLE, alpha=0.60, density=True,
        edgecolor='white', linewidth=0.3, label=f'Défaillants (n={len(d1):,})')

ax.axvline(np.mean(d0), color=BLUE,   linewidth=1.5, linestyle='--',
           label=f'Moy. non-déf. = {np.mean(d0):.2f}')
ax.axvline(np.mean(d1), color=PURPLE, linewidth=1.5, linestyle='--',
           label=f'Moy. déf. = {np.mean(d1):.2f}')

ax.set_title('Distribution de la Distance au Défaut $\\widehat{DD}$ par statut de défaut\n'
             '(le chevauchement explique l\'AUC limitée du modèle de Merton)',
             fontsize=10, color=BLUE, fontweight='bold')
ax.set_xlabel('$\\widehat{DD} = \\hat{d}_2$', fontsize=10)
ax.set_ylabel('Densité', fontsize=9)
ax.legend(fontsize=8, framealpha=0.5)
plt.tight_layout()
plt.savefig('fig_dd_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Construction du vecteur hybride à 16 features ─────────────
# On concatène les 12 ratios comptables + les 4 features Merton.
# Les features Merton ajoutent un signal prospectif  aux ratios rétrospectifs.

X_train_hyb = pd.concat([
    X_train.reset_index(drop=True),
    mf_train[MERTON_COLS].fillna(mf_train[MERTON_COLS].median())
], axis=1)

X_test_hyb = pd.concat([
    X_test.reset_index(drop=True),
    mf_test[MERTON_COLS].fillna(mf_test[MERTON_COLS].median())
], axis=1)

HYBRID_COLS = SELECTED + MERTON_COLS

# Preprocessing complet sur le vecteur hybride
win_h    = fit_winsorise(X_train_hyb)
Xh_tr_w  = apply_winsorise(X_train_hyb, win_h)
Xh_te_w  = apply_winsorise(X_test_hyb, win_h)

imp_h    = SimpleImputer(strategy='median')
Xh_tr_i  = pd.DataFrame(imp_h.fit_transform(Xh_tr_w),  columns=HYBRID_COLS)
Xh_te_i  = pd.DataFrame(imp_h.transform(Xh_te_w),       columns=HYBRID_COLS)

sc_h     = StandardScaler()
Xh_tr_sc = pd.DataFrame(sc_h.fit_transform(Xh_tr_i),  columns=HYBRID_COLS)
Xh_te_sc = pd.DataFrame(sc_h.transform(Xh_te_i),       columns=HYBRID_COLS)

# SMOTE sur le vecteur hybride si disponible
if SMOTE_AVAILABLE:
    smote_h = SMOTE(random_state=42)
    Xh_tr_sm, y_tr_sm = smote_h.fit_resample(Xh_tr_i, y_train.reset_index(drop=True))
    Xh_tr_sm = pd.DataFrame(Xh_tr_sm, columns=HYBRID_COLS)
    sc_h2    = StandardScaler()
    Xh_tr_sm_sc = pd.DataFrame(sc_h2.fit_transform(Xh_tr_sm),  columns=HYBRID_COLS)
    Xh_te_sm_sc = pd.DataFrame(sc_h2.transform(Xh_te_i),        columns=HYBRID_COLS)
else:
    Xh_tr_sm, y_tr_sm = Xh_tr_i.copy(), y_train.reset_index(drop=True).copy()
    Xh_tr_sm_sc = Xh_tr_sc.copy()
    Xh_te_sm_sc = Xh_te_sc.copy()

y_test_arr = y_test.reset_index(drop=True).values

print(f'Vecteur hybride : {len(HYBRID_COLS)} features')
print(f'  Comptables : {SELECTED}')
print(f'  Merton     : {MERTON_COLS}')

---
## 4. Régression Logistique

La Régression Logistique est notre **référence statistique** (baseline). Elle est :
- **linéaire** : elle suppose que le log-odds de défaut est une combinaison linéaire des features
- **interprétable** : chaque coefficient $\beta_j$ mesure la contribution marginale de la variable $j$
- **bien calibrée** : entraînée sur la log-vraisemblance binaire, elle produit des probabilités fiables

On sélectionne le paramètre de régularisation $C = 1/\lambda$ par **validation croisée stratifiée à 5 plis** avec l'AUC-ROC comme critère.

In [ ]:
# ── Cross-validation pour sélectionner C ─────────────────────
# On balaye une grille de valeurs de C (= 1/lambda) en log-échelle.
# La stratification garantit que chaque pli contient ~6.94 % de défauts.

from scipy.stats import loguniform

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grille de recherche : C de 10^-4 à 10^2, les deux pénalités
param_grid_lr = {
    'C'       : np.logspace(-4, 2, 15),
    'penalty' : ['l1', 'l2'],
}

lr_search = RandomizedSearchCV(
    LogisticRegression(solver='saga', max_iter=2000, random_state=42),
    param_distributions=param_grid_lr,
    n_iter=20,             # 20 configurations aléatoires
    scoring='roc_auc',     # AUC-ROC comme critère de sélection
    cv=cv_lr,
    random_state=42,
    n_jobs=-1
)

print('Validation croisée en cours...')
lr_search.fit(Xh_tr_sm_sc, y_tr_sm)

best_C   = lr_search.best_params_['C']
best_pen = lr_search.best_params_['penalty']
best_cv  = lr_search.best_score_

print(f'\nMeilleure configuration :')
print(f'  C = {best_C:.5f} (lambda = {1/best_C:.3f})')
print(f'  Pénalité = {best_pen}')
print(f'  AUC-ROC en CV = {best_cv:.4f}')

In [ ]:
# ── Entraînement final sur tout le train ──────────────────────
# On réentraîne avec les meilleurs hyperparamètres sur l'ensemble
# d'entraînement complet (pas juste sur les folds de CV).

lr_final = LogisticRegression(
    C=best_C, penalty=best_pen,
    solver='saga', max_iter=2000, random_state=42
)
lr_final.fit(Xh_tr_sm_sc, y_tr_sm)

# Prédictions sur le test set (jamais vu pendant l'entraînement)
lr_prob = lr_final.predict_proba(Xh_te_sm_sc)[:, 1]   # probabilités
lr_pred = lr_final.predict(Xh_te_sm_sc)                # classes (seuil 0.5)

print('Régression Logistique — performances sur le test set :')
print(f'  AUC-ROC  : {roc_auc_score(y_test_arr, lr_prob):.4f}')
print(f'  F1-Score : {f1_score(y_test_arr, lr_pred):.4f}')
print(f'  Brier    : {brier_score_loss(y_test_arr, lr_prob):.4f}')

In [ ]:
# ── Coefficients de la Régression Logistique ─────────────────
# Les coefficients (sur données standardisées) mesurent la contribution
# marginale de chaque variable au log-odds de défaut.
# Un coefficient négatif = la variable est protectrice (ex : A10 solvabilité).
# Un coefficient positif = la variable est un facteur de risque (ex : A2 levier).

coef_df = pd.DataFrame({
    'Variable'   : HYBRID_COLS,
    'Coefficient': lr_final.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=True)  # tri par valeur absolue

fig, ax = plt.subplots(figsize=(10, 6))

# Colorer : violet si Merton, bleu si comptable
colors_bar = [PURPLE if v in MERTON_COLS else BLUE for v in coef_df['Variable']]

ax.barh(coef_df['Variable'], coef_df['Coefficient'],
        color=colors_bar, alpha=0.78, edgecolor='white')
ax.axvline(0, color=GRAY, linewidth=0.8, linestyle='-')

ax.set_title('Coefficients de la Régression Logistique (variables standardisées)\n'
             'Violet = features Merton | Bleu = ratios comptables',
             fontsize=10, color=BLUE, fontweight='bold')
ax.set_xlabel('Coefficient', fontsize=9)
ax.tick_params(labelsize=8)
plt.tight_layout()
plt.savefig('fig_lr_coefficients.pdf', bbox_inches='tight')
plt.show()

print('\nTop 5 variables (valeur absolue) :')
print(coef_df.tail(5).to_string(index=False))

---
## 5. XGBoost

XGBoost agrège séquentiellement des centaines d'arbres de décision peu profonds ("weak learners").  
À chaque étape $m$, un nouvel arbre est entraîné sur les **résidus** de l'ensemble courant :

$$f_m(\mathbf{x}) = f_{m-1}(\mathbf{x}) + \eta \cdot T_m(\mathbf{x})$$

Le paramètre `scale_pos_weight` ≈ 13.4 compense le déséquilibre 93%/7% en sur-pondérant la classe des défaillants dans la fonction de perte.

Le réglage des hyperparamètres se fait par **random search** sur 50 configurations, évaluées en 5-fold CV stratifiée.

In [ ]:
# ── Entraînement XGBoost ou substitut RandomForest ────────────

if XGB_AVAILABLE:
    from scipy.stats import randint, uniform

    # Ratio exact pour scale_pos_weight = nb_non_défauts / nb_défauts
    spw = (y_tr_sm == 0).sum() / (y_tr_sm == 1).sum()
    print(f'scale_pos_weight naturel : {spw:.2f}')

    # Espace de recherche des hyperparamètres
    # Chaque configuration est un tirage aléatoire dans ces distributions
    param_dist_xgb = {
        'n_estimators'      : randint(100, 800),        # nombre d'arbres
        'max_depth'         : randint(3, 9),            # profondeur max
        'learning_rate'     : uniform(0.01, 0.29),      # taux d'apprentissage η
        'subsample'         : uniform(0.5, 0.5),        # fraction de lignes par arbre
        'colsample_bytree'  : uniform(0.5, 0.5),        # fraction de colonnes par arbre
        'reg_lambda'        : uniform(0.001, 10),       # régularisation L2
        'scale_pos_weight'  : [1, 5, 10, round(spw)],  # correction du déséquilibre
    }

    xgb_base = xgb.XGBClassifier(
        eval_metric='logloss', random_state=42, verbosity=0)

    search_xgb = RandomizedSearchCV(
        xgb_base, param_dist_xgb,
        n_iter=50,             # 50 configurations aléatoires
        scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        random_state=42, n_jobs=-1
    )

    print('Random search XGBoost (50 configs × 5 plis)...')
    search_xgb.fit(Xh_tr_sm, y_tr_sm)   # XGBoost n'a pas besoin de standardisation

    print('\nMeilleurs hyperparamètres XGBoost :')
    for k, v in search_xgb.best_params_.items():
        print(f'  {k:<22} : {v}')
    print(f'  AUC-ROC en CV        : {search_xgb.best_score_:.4f}')

    # Réentraînement final sur tout le train
    xgb_final = xgb.XGBClassifier(
        **search_xgb.best_params_,
        eval_metric='logloss', random_state=42, verbosity=0
    )
    xgb_final.fit(Xh_tr_sm, y_tr_sm)

else:
    # ── Substitut : RandomForest ──────────────────────────────
    from sklearn.ensemble import RandomForestClassifier
    print('[Substitut] RandomForest à la place de XGBoost')
    xgb_final = RandomForestClassifier(
        n_estimators=300, max_depth=6,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    xgb_final.fit(Xh_tr_sm, y_tr_sm)

# Prédictions
xgb_prob = xgb_final.predict_proba(Xh_te_i)[:, 1]   # sans standardisation
xgb_pred = xgb_final.predict(Xh_te_i)

print('\nXGBoost — performances sur le test set :')
print(f'  AUC-ROC  : {roc_auc_score(y_test_arr, xgb_prob):.4f}')
print(f'  F1-Score : {f1_score(y_test_arr, xgb_pred):.4f}')
print(f'  Brier    : {brier_score_loss(y_test_arr, xgb_prob):.4f}')

In [ ]:
# ── Importance des variables XGBoost ─────────────────────────
# L'importance mesure la réduction totale de la perte attribuable
# aux divisions sur chaque variable à travers tous les arbres.
# On distingue les features Merton (violet) des ratios comptables (bleu)
# pour vérifier si le signal structurel est exploité par le modèle.

if hasattr(xgb_final, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'Variable'   : HYBRID_COLS,
        'Importance' : xgb_final.feature_importances_
    }).sort_values('Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_imp = [PURPLE if v in MERTON_COLS else BLUE for v in imp_df['Variable']]
    ax.barh(imp_df['Variable'], imp_df['Importance'],
            color=colors_imp, alpha=0.80, edgecolor='white')

    ax.set_title('Importance des variables — XGBoost\n'
                 'Violet = features Merton | Bleu = ratios comptables',
                 fontsize=10, color=BLUE, fontweight='bold')
    ax.set_xlabel('Importance (réduction de perte totale)', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(handles=[
        mpatches.Patch(color=PURPLE, alpha=0.8, label='Features Merton'),
        mpatches.Patch(color=BLUE,   alpha=0.8, label='Ratios comptables')
    ], fontsize=9)
    plt.tight_layout()
    plt.savefig('fig_xgb_importance.pdf', bbox_inches='tight')
    plt.show()

    print('\nTop 5 variables :')
    print(imp_df.tail(5).to_string(index=False))

---
## 6. Ablation Study — Quelle est la vraie contribution des features Merton ?

### Problème de la comparaison directe

Comparer Merton (2 inputs : A2, A29) à XGBoost (16 inputs) est **inéquitable** : l'avantage du ML peut venir de l'accès à plus d'information, pas de la méthode elle-même.

### Principe de l'ablation study

On fixe le modèle (Régression Logistique) et on fait varier uniquement les features. Cela permet d'isoler la contribution de chaque source d'information.

| Config | Features | $p$ | Question posée |
|---|---|---|---|
| **C1** | Merton brut $N(-\hat{d}_2)$ | 2 | Score structurel pur |
| **C2** | LR sur 12 ratios comptables | 12 | ML sans Merton |
| **C3** | LR sur 4 features Merton | 4 | ML avec **mêmes** inputs que Merton |
| **C4** | LR sur 16 hybrides | 16 | Hybride complet |

**Comparaison C1 vs C3** → Le ML fait-il mieux que Merton avec les mêmes données ?  
**Comparaison C2 vs C4** → Les features Merton ajoutent-elles quelque chose aux ratios comptables ?

In [ ]:
# ── Utilitaires pour l'ablation study ────────────────────────

def ks_statistic(y_true, y_score):
    """Statistique KS = max(TPR - FPR) sur la courbe ROC."""
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))


def evaluate(config_name, y_true, y_prob, threshold=0.5):
    """Calcule les 4 métriques pour une configuration donnée."""
    return {
        'Configuration' : config_name,
        'p (features)'  : '—',        # rempli manuellement
        'AUC-ROC'       : round(roc_auc_score(y_true, y_prob), 3),
        'F1-Score'      : round(f1_score(y_true, (y_prob >= threshold).astype(int),
                                         zero_division=0), 3),
        'Brier'         : round(brier_score_loss(y_true, y_prob), 3),
        'KS'            : round(ks_statistic(y_true, y_prob), 3),
    }


def preprocess_and_train_lr(X_tr, X_te, y_tr, C=0.1):
    """
    Pipeline complet pour une configuration :
    winsorise → impute → standardise → SMOTE → entraîne LR → prédit.
    Retourne les probabilités prédites sur X_te.
    """
    # Winsorisation
    wb = fit_winsorise(X_tr)
    Xw_tr = apply_winsorise(X_tr, wb)
    Xw_te = apply_winsorise(X_te, wb)

    # Imputation
    imp = SimpleImputer(strategy='median')
    Xi_tr = pd.DataFrame(imp.fit_transform(Xw_tr), columns=Xw_tr.columns)
    Xi_te = pd.DataFrame(imp.transform(Xw_te),     columns=Xw_te.columns)

    # SMOTE (si disponible)
    if SMOTE_AVAILABLE:
        Xi_tr, y_tr = SMOTE(random_state=42).fit_resample(Xi_tr, y_tr)
        Xi_tr = pd.DataFrame(Xi_tr, columns=Xw_tr.columns)

    # Standardisation
    sc = StandardScaler()
    Xs_tr = sc.fit_transform(Xi_tr)
    Xs_te = sc.transform(Xi_te)

    # Entraînement LR avec hyperparamètres fixes pour comparabilité
    lr = LogisticRegression(C=C, max_iter=2000,
                            random_state=42, class_weight='balanced')
    lr.fit(Xs_tr, y_tr)
    return lr.predict_proba(Xs_te)[:, 1]


print('Utilitaires ablation study prêts.')

In [ ]:
# ── RÈGLE CRITIQUE : features Merton recalculées après le split ─
# On utilise X_train et X_test (les matrices brutes avant preprocessing)
# pour recalculer les features Merton séparément sur chaque ensemble.
# Cela garantit l'absence de toute fuite de données.

print('Recalcul des features Merton (train et test séparés)...')

mf_tr_abl = (
    df.loc[X_train.index, SELECTED]
    .apply(compute_merton_row, axis=1, result_type='expand')
    .rename(columns={'DD':'DD','PD':'PD','sigma_V':'sigma_V','spread':'spread'})
    .reset_index(drop=True)
)
mf_te_abl = (
    df.loc[X_test.index, SELECTED]
    .apply(compute_merton_row, axis=1, result_type='expand')
    .rename(columns={'DD':'DD','PD':'PD','sigma_V':'sigma_V','spread':'spread'})
    .reset_index(drop=True)
)

y_train_abl = y_train.reset_index(drop=True)
X_train_abl = X_train.reset_index(drop=True)
X_test_abl  = X_test.reset_index(drop=True)

print(f'Features Merton (ablation) — train : {mf_tr_abl.shape}, test : {mf_te_abl.shape}')

In [ ]:
# ── Exécution des 4 configurations ───────────────────────────

ablation_results = []

# ── Config 1 : Merton standalone ─────────────────────────────
# On utilise directement PD = N(-DD) comme score, sans apprentissage.
prob_c1 = mf_te_abl['PD'].fillna(0.5).values
r1 = evaluate('C1 — Merton standalone', y_test_arr, prob_c1)
r1['p (features)'] = 2  # A2 et A29 seulement
ablation_results.append(r1)
print(f'Config 1 terminée — AUC : {r1["AUC-ROC"]}')

# ── Config 2 : LR sur 12 ratios comptables ───────────────────
# LR n'a accès qu'aux ratios financiers, sans les features Merton.
prob_c2 = preprocess_and_train_lr(X_train_abl, X_test_abl, y_train_abl)
r2 = evaluate('C2 — LR 12 ratios comptables', y_test_arr, prob_c2)
r2['p (features)'] = 12
ablation_results.append(r2)
print(f'Config 2 terminée — AUC : {r2["AUC-ROC"]}')

# ── Config 3 : LR sur 4 features Merton ──────────────────────
# LR a accès aux mêmes features que le modèle de Merton,
# mais les apprend plutôt que d'utiliser la formule analytique.
prob_c3 = preprocess_and_train_lr(mf_tr_abl, mf_te_abl, y_train_abl)
r3 = evaluate('C3 — LR 4 features Merton', y_test_arr, prob_c3)
r3['p (features)'] = 4
ablation_results.append(r3)
print(f'Config 3 terminée — AUC : {r3["AUC-ROC"]}')

# ── Config 4 : LR sur 16 features hybrides ───────────────────
# LR a accès aux 12 ratios + 4 features Merton.
X_hyb_tr = pd.concat([X_train_abl, mf_tr_abl[MERTON_COLS].fillna(mf_tr_abl[MERTON_COLS].median())], axis=1)
X_hyb_te = pd.concat([X_test_abl,  mf_te_abl[MERTON_COLS].fillna(mf_te_abl[MERTON_COLS].median())], axis=1)
prob_c4 = preprocess_and_train_lr(X_hyb_tr, X_hyb_te, y_train_abl)
r4 = evaluate('C4 — LR 16 hybrides', y_test_arr, prob_c4)
r4['p (features)'] = 16
ablation_results.append(r4)
print(f'Config 4 terminée — AUC : {r4["AUC-ROC"]}')

In [ ]:
# ── Tableau de synthèse ───────────────────────────────────────

abl_df = pd.DataFrame(ablation_results)
print('\nABLATION STUDY — TABLEAU DE SYNTHÈSE')
print('='*70)
print(abl_df.to_string(index=False))
print()

# Trois constats clés
print('TROIS CONSTATS EMPIRIQUES :')
print()
gain_c3_c1 = (r3['AUC-ROC'] - r1['AUC-ROC']) * 100
print(f'  C1 — Le ML ajoute-t-il quelque chose avec les mêmes inputs que Merton ?')
print(f'       C1 (Merton brut)     AUC = {r1["AUC-ROC"]:.3f}')
print(f'       C3 (LR 4 Merton)    AUC = {r3["AUC-ROC"]:.3f}')
print(f'       Gain : {gain_c3_c1:+.1f} pp → la formule de Merton est quasi optimale sur ses propres inputs')
print()
gain_c4_c2 = (r4['AUC-ROC'] - r2['AUC-ROC']) * 100
print(f'  C2 — Les features Merton ajoutent-elles qqch aux ratios comptables ?')
print(f'       C2 (LR 12 compt.)   AUC = {r2["AUC-ROC"]:.3f}')
print(f'       C4 (LR 16 hybrides) AUC = {r4["AUC-ROC"]:.3f}')
print(f'       Gain : {gain_c4_c2:+.1f} pp → features Merton redondantes avec A2, A29 déjà présents')
print()
print(f'  C3 — Meilleur feature set ?')
best = max(ablation_results, key=lambda x: x['AUC-ROC'])
print(f'       {best["Configuration"]} — AUC = {best["AUC-ROC"]:.3f}')
print(f'       Les ratios comptables seuls dominent toutes les configurations.')

In [ ]:
# ── Visualisation : barres groupées de l'ablation study ───────
# Le graphique en barres horizontales est le choix recommandé pour
# le classement et la comparaison de performances.

metrics_abl = ['AUC-ROC', 'F1-Score', 'KS']
labels = ['C1\nMerton brut', 'C2\nLR 12 compt.', 'C3\nLR 4 Merton', 'C4\nLR 16 hybr.']
palette = [GRAY, BLUE, '#8830b0', PURPLE]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, metrics_abl):
    values = [r[metric] for r in ablation_results]
    bars   = ax.barh(range(4), values, color=palette,
                     alpha=0.82, edgecolor='white', linewidth=0.5)

    # Afficher la valeur à droite de chaque barre
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', ha='left',
                fontsize=9, fontweight='bold', color=GRAY)

    ax.set_yticks(range(4))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title(metric, fontsize=11, color=BLUE, fontweight='bold')
    ax.set_xlim([0, max(values) * 1.25])
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Ablation Study — Même modèle (LR), features différentes\n'
    'Isole la vraie contribution du modèle de Merton',
    fontsize=11, color=BLUE, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('fig_ablation_study.pdf', bbox_inches='tight')
plt.show()

---
## 7. Comparaison finale des 3 modèles

On compare maintenant les trois modèles dans leur configuration **naturelle** :
- **Merton** : score brut $N(-\hat{d}_2)$ sur 2 inputs
- **Régression Logistique** : 16 features hybrides, hyperparamètres optimisés par CV
- **XGBoost** : 16 features hybrides, random search sur 50 configurations

Les 4 métriques capturent des dimensions différentes :
- **AUC-ROC** : capacité de classement (indépendant du seuil)
- **F1-Score** : équilibre précision/rappel au seuil 0.5
- **Brier Score** : calibration des probabilités (essentielle pour l'ECL et le SCR)
- **KS Statistic** : séparation maximale entre les deux distributions de scores

In [ ]:
# ── Tableau de performances final ────────────────────────────

def ks_stat(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

results_final = pd.DataFrame([
    {
        'Modèle'   : 'Merton (structural)',
        'AUC-ROC'  : round(roc_auc_score(y_test_arr, merton_prob_test), 3),
        'F1-Score' : round(f1_score(y_test_arr, (merton_prob_test >= 0.5).astype(int)), 3),
        'Brier'    : round(brier_score_loss(y_test_arr, merton_prob_test), 3),
        'KS'       : round(ks_stat(y_test_arr, merton_prob_test), 3),
    },
    {
        'Modèle'   : 'Régression Logistique',
        'AUC-ROC'  : round(roc_auc_score(y_test_arr, lr_prob), 3),
        'F1-Score' : round(f1_score(y_test_arr, lr_pred), 3),
        'Brier'    : round(brier_score_loss(y_test_arr, lr_prob), 3),
        'KS'       : round(ks_stat(y_test_arr, lr_prob), 3),
    },
    {
        'Modèle'   : 'XGBoost',
        'AUC-ROC'  : round(roc_auc_score(y_test_arr, xgb_prob), 3),
        'F1-Score' : round(f1_score(y_test_arr, xgb_pred), 3),
        'Brier'    : round(brier_score_loss(y_test_arr, xgb_prob), 3),
        'KS'       : round(ks_stat(y_test_arr, xgb_prob), 3),
    },
])

print('TABLEAU DE PERFORMANCES FINAL')
print('='*55)
print(results_final.to_string(index=False))

In [ ]:
# ── Courbes ROC superposées ───────────────────────────────────
# La courbe ROC montre le compromis TPR/FPR à tous les seuils.
# Un bon modèle tend vers le coin supérieur gauche.

fig, ax = plt.subplots(figsize=(7, 7))

for name, probs, color, lw in [
    ('Merton',               merton_prob_test, GRAY,   1.8),
    ('Régression Logistique', lr_prob,         BLUE,   2.0),
    ('XGBoost',               xgb_prob,        PURPLE, 2.4),
]:
    fpr, tpr, _ = roc_curve(y_test_arr, probs)
    auc = roc_auc_score(y_test_arr, probs)
    ax.plot(fpr, tpr, color=color, linewidth=lw,
            label=f'{name} (AUC = {auc:.3f})')

# Diagonale = classifieur aléatoire
ax.plot([0, 1], [0, 1], '--', color=GRAY, linewidth=0.9, alpha=0.6,
        label='Aléatoire (AUC = 0.500)')

# Grille légère
for v in [0.2, 0.4, 0.6, 0.8]:
    ax.axhline(v, color=GRAY, linewidth=0.3, alpha=0.4)
    ax.axvline(v, color=GRAY, linewidth=0.3, alpha=0.4)

ax.set_xlabel('Taux de faux positifs (FPR)', fontsize=10)
ax.set_ylabel('Taux de vrais positifs (TPR)', fontsize=10)
ax.set_title(f'Courbes ROC — Ensemble de test ($n = {len(y_test_arr):,}$)',
             fontsize=11, color=BLUE, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.6)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig('fig_roc_final.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Distributions des scores par statut de défaut ─────────────
# Un bon modèle concentre les non-défaillants près de 0
# et les défaillants près de 1. Le chevauchement mesure la difficulté.

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (title, probs, color) in zip(axes, [
    ('Merton ($\\widehat{PD}$)',   merton_prob_test, GRAY),
    ('Régression Logistique',       lr_prob,          BLUE),
    ('XGBoost',                     xgb_prob,         PURPLE),
]):
    s0 = probs[y_test_arr == 0]   # scores des non-défaillants
    s1 = probs[y_test_arr == 1]   # scores des défaillants
    bins = np.linspace(0, 1, 25)

    ax.hist(s0, bins=bins, color=BLUE,   alpha=0.55, density=True,
            edgecolor='white', linewidth=0.3, label=f'Non-déf. (n={len(s0):,})')
    ax.hist(s1, bins=bins, color=PURPLE, alpha=0.60, density=True,
            edgecolor='white', linewidth=0.3, label=f'Déf. (n={len(s1):,})')

    # Lignes des moyennes
    ax.axvline(s0.mean(), color=BLUE,   linewidth=1.4, linestyle='--')
    ax.axvline(s1.mean(), color=PURPLE, linewidth=1.4, linestyle='--')

    ax.set_title(title, fontsize=10, color=color, fontweight='bold')
    ax.set_xlabel('Score prédit', fontsize=9)
    ax.set_ylabel('Densité', fontsize=9)
    ax.legend(fontsize=8, framealpha=0.5)
    ax.set_xlim([0, 1])

plt.suptitle('Distributions des scores prédits par statut de défaut\n'
             'Un bon modèle maximise la séparation entre les deux distributions',
             fontsize=11, color=BLUE, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_score_distributions.pdf', bbox_inches='tight')
plt.show()

---
## 8. Sauvegarde des résultats

On exporte :
- `predictions_test.csv` : scores des 3 modèles sur le test set (pour traçabilité)
- `results_final.csv` : tableau des 4 métriques (à intégrer dans le rapport)
- `ablation_study.csv` : résultats de l'ablation study
- Les figures PDF sont déjà sauvegardées au fil du notebook

In [ ]:
# ── Export des données ────────────────────────────────────────

# Prédictions des 3 modèles sur le test set
pd.DataFrame({
    'y_true'        : y_test_arr,
    'merton_prob'   : merton_prob_test,
    'lr_prob'       : lr_prob,
    'xgb_prob'      : xgb_prob,
}).to_csv('predictions_test.csv', index=False)

# Tableau de performances
results_final.to_csv('results_final.csv', index=False)

# Ablation study
pd.DataFrame(ablation_results).to_csv('ablation_study.csv', index=False)

print('Fichiers sauvegardés :')
print('  predictions_test.csv      — scores des 3 modèles sur le test')
print('  results_final.csv         — tableau de performances (4 métriques)')
print('  ablation_study.csv        — résultats de l\'ablation study')
print()
print('Figures PDF sauvegardées :')
print('  fig_dd_distribution.pdf   — distribution DD par statut')
print('  fig_lr_coefficients.pdf   — coefficients LR')
print('  fig_xgb_importance.pdf    — importance variables XGBoost')
print('  fig_ablation_study.pdf    — ablation study (barres groupées)')
print('  fig_roc_final.pdf         — courbes ROC')
print('  fig_score_distributions.pdf — distributions des scores')